In [1]:
import io
import zipfile
import urllib.request
from pathlib import Path
import pandas as pd

# --- Params ---
participant_id = 'MW_SSH_05'
transcripts_dir = Path('transcripts')  # extracted CSVs will be placed here
transcripts_dir.mkdir(parents=True, exist_ok=True)

# --- Ensure TRANSCRIPTS.zip is downloaded & extracted ---
zip_url = 'https://zenodo.org/records/15222484/files/TRANSCRIPTS.zip?download=1'
expected_csv = transcripts_dir / f'{participant_id}.csv'

if not expected_csv.exists():
    print('Downloading TRANSCRIPTS.zip from Zenodo...')
    with urllib.request.urlopen(zip_url) as resp:
        data = resp.read()
    with zipfile.ZipFile(io.BytesIO(data)) as zf:
        zf.extractall(transcripts_dir)
    print(f'Extracted transcripts to: {transcripts_dir.resolve()}')
else:
    print('Transcripts already present, skipping download.')

# --- Load problems and responses from Zenodo ---
problems = pd.read_csv('https://zenodo.org/records/15222484/files/PROBLEMS.csv')
responses = pd.read_csv('https://zenodo.org/records/15222484/files/PROBLEMS_RESPONSES.csv')
merged_problems = pd.merge(responses, problems, on='problem_id', how='inner')

# Use column names as defined in the dataset docs
prediction_features = ['participant_decision', 'participant_certaintity', 'model_class', 'model_probability']
print(merged_problems[merged_problems['participant_id'] == participant_id][prediction_features])

# --- Retrieve transcript text for problem __P1__ for the participant ---
transcripts = pd.read_csv(expected_csv)
# Forward-fill problem markers within the participant's transcript
if 'problem_id' in transcripts.columns:
    transcripts['problem_id'] = transcripts['problem_id'].ffill()
    subset = transcripts[transcripts['problem_id'] == '__P1__']
else:
    subset = pd.DataFrame()

# Show key columns if present
cols = [c for c in ['speaker_id', 'slide_id', 'question_id', 'problem_id', 'text'] if c in subset.columns]
print(subset[cols] if cols else subset)

Transcripts already present, skipping download.
   participant_decision participant_certaintity model_class model_probability
57              trujący           średnio pewny     jadalny              0,54
58              jadalny           średnio pewny     jadalny              0,95
59              jadalny      zdecydowanie pewny     trujący                 1
    speaker_id slide_id question_id problem_id  \
180  MW_SSH_05      NaN         NaN     __P1__   
181         MW      NaN         NaN     __P1__   
182  MW_SSH_05      NaN         NaN     __P1__   

                                                  text  
180        Ok. Średnica 17,30... Mogę po całej kartce?  
181               Jasne, proszę sobie mazać spokojnie.  
182  Ok to wróćmy do… To była wysokość, średnica ka...  


In [ ]:
from LocalLLM import LocalLLM
from TranscriptParser import TranscriptParser

parser = TranscriptParser()
parsed_transcript = parser.parse_file_grouped_by_slide(expected_csv)
slides = list(parsed_transcript)

local_llm = LocalLLM()
print(local_llm.ask(
  """Potrzebuję abyś przeanalizował tekst i skupił się na wydobyciu profilu uczestnika na bazie dynamiki jego wypowiedzi:
  -długość wypowedzi, 
  -reakcja na zadania, 
  -sposób wypowiedzi.
  Odpowiedz w formacie json:
  {
  "participant_id": "string",
  "profile": {
    "engagement_level": "low/medium/high",
    "confidence_level": "low/medium/high",
    "communication_style": "concise/detailed/ambiguous"
  }
  """,
    context="\n\n".join(f"Slajd {r['slide_id']} - {r['participant_id']}: {r['aggregated_text']}" for r in slides)))

[{'source_file': 'MW_SSH_05.csv', 'slide_id': '__S00__', 'participant_id': 'MW', 'aggregated_text': 'Super. Czyli ten folder testowy możemy sobie zamknąć. I teraz prosiłabym pana o wejście do tego folderu właśnie badanie. I tu nas interesuje przede wszystkim ten trzeci plik, czyli prezentacja multimedialna. To jest ten plik nasz właściwy, na którym będziemy sobie działać. Można tutaj włączyć tą opcję takiego edytowania, gdyby nam była potrzebna, czyli na żółtym paseczku tutaj kliknąć. I teraz tak, chciałabym żeby pan tak na razie szybkim rzutem oka przejrzał tę prezentację tak zupełnie swobodnie. Powiedział mi… Można, jak najbardziej. Komentowanie jest bardzo mile widziane. Komentując wszystko, co pan widzi, do czego ta prezentacja może służyć, czy ona jest raczej ładna, czy nieładna, czy neutralna, wszystko co się panu rzuci w oczy tak na pierwszy rzut oka. To możemy sobie iść tak od początku prezentacji albo w jakimkolwiek innym też układzie, jak panu by odpowiadał. Generalnie naszym

In [20]:
output_path = str(Path.home() / 'xaifungiD-analysis' / 'data' / 'parsed_transcripts.jsonl')
parser.parse_all(input_dir=transcripts_dir, output_path=output_path)

{'files': 38, 'rows': 1781}